Grover's algorithm.
Before even having a functional quantum computer, many algorithms were established to prove the quantum advantage. Among them we find the famous Grover's algorithm. The problem is simple , can we locate a key by making a query in an unstructered database ? In the classical way , we just loop over all the items and comparing it to the key , until we found it , worst case is searching in the whole database , and in average you need N/2 queries. 
So classicaly the algorithm is like this : retreive element , compare to key : if equal stop if not continue to next element , let's note that we suppose there is only one key in the whole database to make things simple.

In [ ]:
%pip install matplotlib qiskit-aer

  Using cached qiskit_aer-0.18.2-cp314-cp314-win_amd64.whl.metadata (8.9 kB)
Using cached qiskit_aer-0.18.2-cp314-cp314-win_amd64.whl (10.0 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
#initialization
import matplotlib.pyplot as plt
import numpy as np
import math

# importing Qiskit
from qiskit import transpile 
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister

# import basic plot tools
from qiskit.visualization import plot_histogram

In [74]:
n = 3
grover_circuit = QuantumCircuit(n)

In [75]:
def initialize_s(qc , qubits):
 for q in qubits:
    qc.h(q)
 return qc

 

In [76]:
grover_circuit = initialize_s(grover_circuit, range(n))

Let the marked solution be the three-qubit computational-basis state $|011\\rangle$. The Boolean oracle is $f(x)=1$ for $x=011$ and $f(x)=0$ otherwise. Grover's phase oracle applies a phase of $-1$ to the marked state and leaves all other basis states unchanged:

$$O_f|x\\rangle=(-1)^{f(x)}|x\\rangle.$$

In the basis $(|000\\rangle,|001\\rangle,|010\\rangle,|011\\rangle,|100\\rangle,|101\\rangle,|110\\rangle,|111\\rangle)$, its matrix is the $8\\times 8$ diagonal matrix

$$O_f=\\operatorname{diag}(1,1,1,-1,1,1,1,1).$$

The $-1$ is the fourth diagonal entry because $|011\\rangle$ is the fourth state in this basis.

In [77]:
def oracle (qc) :
 qc.x(0)
 qc.x(1)
 qc.h(2)
 qc.ccx(0, 1, 2)  
 qc.h(2)
 qc.x(0)
 qc.x(1)
 return qc

 

In [78]:
grover_circuit = oracle(grover_circuit)



In [79]:
grover_circuit.draw()

┌───┐┌───┐     ┌───┐
q_0: ┤ H ├┤ X ├──■──┤ X ├
     ├───┤├───┤  │  ├───┤
q_1: ┤ H ├┤ X ├──■──┤ X ├
     ├───┤├───┤┌─┴─┐├───┤
q_2: ┤ H ├┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘└───┘

In [80]:
def diffuser(qc , n=3):
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

In [81]:
grover_circuit = diffuser(grover_circuit)

In [82]:
grover_circuit.draw()


┌───┐┌───┐     ┌───┐┌───┐┌───┐          ┌───┐┌───┐     
q_0: ┤ H ├┤ X ├──■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤  │  ├───┤├───┤├───┤       │  ├───┤├───┤     
q_1: ┤ H ├┤ X ├──■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤┌─┴─┐├───┤├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐
q_2: ┤ H ├┤ H ├┤ X ├┤ H ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘

In [83]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

sim = AerSimulator()
grover_circuit.measure_all()
compiled = transpile(grover_circuit, sim)

job = sim.run(compiled, shots=1024)
counts = job.result().get_counts()
print(counts)

plot_histogram(counts, title="Grover result")
plt.show()

{'001': 34, '100': 801, '010': 36, '111': 36, '000': 30, '110': 34, '011': 23, '101': 30}
